In [ ]:
import pandas as pd

df = pd.read_csv("../data/WA_Fn-UseC_-Telco-Customer-Churn.csv")
print(df.shape)
df.head()

The "Telco Customer Churn" dataset contains 21 columns and 7043 rows.

In [ ]:
df.info()

None of the features contain null values. "SeniorCitizen", "tenure" and "MonthlyCharges" are numeric while all the other features are strings (categorical). "TotalCharges" are of the type string instead of numeric values.

In [ ]:
df["TotalCharges"] = pd.to_numeric(df["TotalCharges"], errors="coerce")
# "coerce" tells pandas to convert any invalid value to NaN
print(df["TotalCharges"].isna().sum())
df[df["TotalCharges"].isna()][["customerID", "tenure", "MonthlyCharges", "TotalCharges", "Churn"]]

After converting all "TotalCharges" to numeric form we find 11 rows without any data. All these rows contains 0 as tenure meaning they're new customers meaning they haven't been charged yet.

In [ ]:
print(df.isna().sum())
print("Duplicate rows:", df.duplicated().sum())
print("Duplicate IDs:", df["customerID"].duplicated().sum())

No other missing values apart from the "TotalCharges" of new customers mentioned earlier. Handling of duplication values isn't required either.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

print(df["Churn"].value_counts())
print(df["Churn"].value_counts(normalize=True).round(3))

sns.countplot(data=df, x="Churn")
plt.title("Churn distribution")
plt.show()

5174 customers stayed while 1869 left. When taken as a proportion trimmed to three decimals, 0.735 (73.5%) stayed while, 0.265 (26.5%) left. The fact that the distribution is not balanced should be taken to consideration as it affects accuracy in the following steps. (for an example a model predicting "No" for everything will have an accuracy of 73.5%)

In [ ]:
num_cols = ["tenure", "MonthlyCharges", "TotalCharges"] # list of numeric columns to be examined

fig, axes = plt.subplots(1, 3, figsize=(15, 4)) # defines canvas with three plots
for ax, col in zip(axes, num_cols): # loops through each numeric column creating respective visualization
    sns.histplot(data=df, x=col, hue="Churn", kde=True, element="step", ax=ax) # draws the histogram for each column, splitting data by color based on whether they churned or not, adding smooth curve using 'kde', and making sure both groups are visible irrespective of them overlapping or not using "step"
    ax.set_title(f"{col} by Churn")
plt.tight_layout() #adjusts the spacing between the plots
plt.show()

df.groupby("Churn")[num_cols].median()

The histograms show that churners have shorter tenure, customers on cheaper plans are less likely to churn and churners have lower total charges, but this is likely a consequence of shorter tenure.

These values are to be scaled accordingly as they have very different ranges ("tenure" from 0 to ~70 vs "TotalCharges" from 0 to ~8,000) as KNN and Logistic Regression are sensitive to feature scale.

In [ ]:
cat_cols = [c for c in df.columns if c not in num_cols + ["customerID", "Churn"]] # list of categorical columns to be examined excluding "customerID" and "Churn"
overall = (df["Churn"] == "Yes").mean() # calculates average churn rate

fig, axes = plt.subplots(4, 4, figsize=(20, 16)) # defines canvas with sixteen plots
for ax, col in zip(axes.flat, cat_cols):
    rate = (df["Churn"] == "Yes").groupby(df[col]).mean().sort_values(ascending=False) # calculates how many churned in each group
    sns.barplot(x=rate.index, y=rate.values, ax=ax) # draws bar chart for each category
    ax.axhline(overall, color="red", linestyle="--") # marls average churn for each chart
    ax.set_title(col)
    ax.set_ylabel("Churn rate")
    ax.tick_params(axis="x", rotation=30) #ensures labels don't overlap
plt.tight_layout()
plt.show()

"Contract" is the strongest categorical indicator showing that customers with no long term plans leave more often. Those with fiber optics seem to churn more than other "InternetService"s. Electronic check users churn about double the other methods. Customers without "TechSupport" or "OnlineSecurity" seem to churn more. Similarly senior citizens and customers without dependents churn more whereas "gender" doesn't seem to have a major effect.

 All these features will require encoding as they are categorical text.

In [ ]:
df_corr = df.copy()
df_corr["Churn"] = (df_corr["Churn"] == "Yes").astype(int) # converts "Churn" to 1/0 so it can be correlated

corr = df_corr[["tenure", "MonthlyCharges", "TotalCharges", "SeniorCitizen", "Churn"]].corr() #create correlation matrix

plt.figure(figsize=(6, 5))
sns.heatmap(corr, annot=True, cmap="coolwarm", fmt=".2f", center=0) # draws a correlation matrix as a color based grid, where 'annot' writes the values inside each cell, and center tells seaborn that the middle of the color scale is 0
plt.title("Correlation matrix")
plt.show()

"tenure" has a negative correlation with "Churn" meaning customers who stay longer are less likely to leave. "MonthlyCharges" has a weak positive correlation meaning customers who pay more churn slightly more. "TotalCharges" and "tenure" are strongly correlated because total charges accumulate over time. "TotalCharges" is also correlated with "MonthlyCharges". This is multicollinearity because "TotalCharges" mostly repeats information already in "tenure" and "MonthlyCharges".

Logistic Regression can become unstable when features are highly correlated. Random Forest is not as much as affected by correlated features.


In [ ]:
df["TotalCharges"] = df["TotalCharges"].fillna(0)

The total charges of new customers can be safely converted to zero as they haven't been charged yet.

In [ ]:
df = df.drop(columns=["customerID"])

"customerID" is just a unique identifier column and provides no insight. Therefore, it can be dropped.

In [ ]:
service_cols = ["OnlineSecurity", "OnlineBackup", "DeviceProtection",
                "TechSupport", "StreamingTV", "StreamingMovies"]
df[service_cols] = df[service_cols].replace("No internet service", "No")
df["MultipleLines"] = df["MultipleLines"].replace("No phone service", "No")

Since "No internet services" mean the same as "No", they will be replaced. Similarly, "No phone service" entries of "MultipleLines" will be replaced by "no" too.

In [ ]:
df["Churn"] = (df["Churn"] == "Yes").astype(int) #converts to 1s and 0s

binary_cols = ["Partner", "Dependents", "PhoneService", "PaperlessBilling",
               "MultipleLines"] + service_cols
for col in binary_cols:
    df[col] = (df[col] == "Yes").astype(int)

df["gender"] = (df["gender"] == "Male").astype(int)

The target column and binary columns are encoded as 1s and 0s.

In [ ]:
df = pd.get_dummies(df, columns=["InternetService", "Contract", "PaymentMethod"], drop_first=True, dtype=int) #performs one hot encoding on multicategory columns where "drop_first=True" drops first alphabetical category from each encoded column.

One hot encoding is performed for columns with multiple categories.